# iter_3 Macro Autoresearch Analysis

Validation CAGR is the selection objective subject to no ruin. Locked OOS and DBMF excess are diagnostics only.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
results = pd.read_csv(Path('results.tsv'), sep='\t')
numeric_cols = ['train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf']
for col in numeric_cols:
    results[col] = pd.to_numeric(results[col], errors="coerce")
results['trial'] = range(1, len(results) + 1)
results['ruined_flag'] = results['ruined'].astype(str).str.lower().isin(['true', '1'])
eligible = results[(~results['ruined_flag']) & (results['validation_cagr'] > 0)].copy()
best = eligible.loc[eligible['validation_cagr'].idxmax()]
keeps = results[results['status'] == 'keep'].copy()
summary = pd.DataFrame([{'rows': len(results), 'keeps': len(keeps), 'ruined': int(results['ruined_flag'].sum()), 'best_idea_id': best['idea_id'], 'best_validation_cagr': best['validation_cagr'], 'best_train_cagr': best['train_cagr'], 'best_oos_cagr': best['oos_cagr'], 'best_benchmark_oos_cagr': best['benchmark_oos_cagr'], 'best_excess_oos_cagr_vs_dbmf': best['excess_oos_cagr_vs_dbmf']}])
summary


In [ ]:
display_cols = ['trial', 'idea_id', 'status', 'train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf', 'ruined', 'description']
results.sort_values('validation_cagr', ascending=False)[display_cols].head(12)


In [ ]:
plot = results[['trial', 'idea_id', 'status', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr']].copy()
plot['running_best_validation_cagr'] = plot['validation_cagr'].where(~results['ruined_flag']).cummax()
fig, ax = plt.subplots(figsize=(12, 7))
colors = plot['status'].map({'keep': '#2ca02c', 'discard': '#7f7f7f', 'ruined': '#d62728', 'invalid_rationale': '#ff7f0e'}).fillna('#7f7f7f')
ax.scatter(plot['trial'], plot['validation_cagr'], c=colors, s=54, label='trial validation CAGR')
ax.plot(plot['trial'], plot['running_best_validation_cagr'], color='#1f77b4', linewidth=2.5, label='running best validation CAGR')
ax.axhline(float(best['benchmark_oos_cagr']), color='#9467bd', linestyle='--', linewidth=1.5, label='DBMF locked-OOS CAGR diagnostic')
ax.scatter([int(best['trial'])], [float(best['validation_cagr'])], color='gold', edgecolor='black', s=180, zorder=5, label='best validation-selected run')
ax.annotate(str(best['idea_id']), xy=(int(best['trial']), float(best['validation_cagr'])), xytext=(int(best['trial']) - 12, float(best['validation_cagr']) + 0.035), arrowprops={'arrowstyle': '->', 'color': 'black'}, fontsize=9)
ax.set_title('iter_3 validation-CAGR progress; OOS/DBMF shown only as diagnostics')
ax.set_xlabel('ledger trial')
ax.set_ylabel('CAGR')
ax.grid(True, alpha=0.25)
ax.legend(loc='best')
fig.tight_layout()
fig.savefig('progress.png', dpi=180)
plt.show()


## Best run interpretation

The selected run is the highest non-ruined validation-CAGR strategy in the ledger. OOS CAGR and DBMF excess are locked diagnostics, not the selection criterion.


In [ ]:
best[['idea_id', 'title', 'mechanism', 'expected_assets', 'feature_inputs', 'human_notes', 'train_cagr', 'validation_cagr', 'oos_cagr', 'benchmark_oos_cagr', 'excess_oos_cagr_vs_dbmf', 'ruined', 'status']].to_frame('best_run')
